# RQ2 Vulnerability Detection Analysis

**Purpose**: Analyze vulnerability detection results from RQ2 experiments comparing Dual-Agent vs Multi-Agent approaches.

**Datasets**: 16 vulnerability detection experiments (8 pods × 2 experiments per pod)
- **Dual-Agent (DA)**: 2-agent approach (Security Researcher + Moderator)
- **Multi-Agent (MA)**: 4-agent approach (Security Researcher + Code Author + Moderator + Review Board)

**Research Questions**:
- **RQ2.1**: How do dual-agent and multi-agent approaches compare in vulnerability detection accuracy?
- **RQ2.2**: What is the impact of prompting strategy (zero-shot vs few-shot)?
- **RQ2.3**: How do model size (4B vs 30B) and reasoning capabilities (Instruct vs Thinking) affect performance?
- **RQ2.4**: What are the energy efficiency tradeoffs between different configurations?

**Date**: November 17, 2025

## 1. Setup & Data Loading

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import warnings
warnings.filterwarnings('ignore')

# Set style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 8)
plt.rcParams['font.size'] = 11

# Paths
PROJECT_ROOT = Path.cwd().parent
RESULTS_DIR = PROJECT_ROOT / 'results'
OUTPUT_DIR = PROJECT_ROOT / 'results' / 'analysis' / 'rq2'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"📂 Project Root: {PROJECT_ROOT}")
print(f"📊 Results Directory: {RESULTS_DIR}")
print(f"💾 Output Directory: {OUTPUT_DIR}")

## 2. Load RQ2 Results from All Pods

Pod configuration:
- **Pod 1**: 4B-Instruct, Zero-Shot
- **Pod 2**: 4B-Thinking, Zero-Shot  
- **Pod 3**: 4B-Instruct, Few-Shot
- **Pod 4**: 4B-Thinking, Few-Shot
- **Pod 5**: 30B-Instruct, Zero-Shot
- **Pod 6**: 30B-Thinking, Zero-Shot
- **Pod 7**: 30B-Instruct, Few-Shot
- **Pod 8**: 30B-Thinking, Few-Shot

In [ ]:
# Define pod configurations
pod_configs = {
    'pod1': {'model_size': '4B', 'model_type': 'Instruct', 'prompting': 'Zero-shot'},
    'pod2': {'model_size': '4B', 'model_type': 'Thinking', 'prompting': 'Zero-shot'},
    'pod3': {'model_size': '4B', 'model_type': 'Instruct', 'prompting': 'Few-shot'},
    'pod4': {'model_size': '4B', 'model_type': 'Thinking', 'prompting': 'Few-shot'},
    'pod5': {'model_size': '30B', 'model_type': 'Instruct', 'prompting': 'Zero-shot'},
    'pod6': {'model_size': '30B', 'model_type': 'Thinking', 'prompting': 'Zero-shot'},
    'pod7': {'model_size': '30B', 'model_type': 'Instruct', 'prompting': 'Few-shot'},
    'pod8': {'model_size': '30B', 'model_type': 'Thinking', 'prompting': 'Few-shot'},
}

# Load all vulnerability detection results
all_results = []

for pod_name, config in pod_configs.items():
    pod_dir = RESULTS_DIR / f'runpod_rq2_{pod_name}'
    
    # Handle nested results directories for some pods
    if (pod_dir / 'results').exists():
        pod_dir = pod_dir / 'results'
    
    # Find DA and MA vulnerability detection files
    da_files = list(pod_dir.glob('DA-vuln-*_classification_report.csv'))
    ma_files = list(pod_dir.glob('MA-vuln-*_classification_report.csv'))
    
    for file in da_files + ma_files:
        agent_type = 'Dual-Agent' if 'DA-' in file.name else 'Multi-Agent'
        
        # Load classification report
        df = pd.read_csv(file)
        
        # Extract metrics (assuming weighted avg row)
        weighted_avg = df[df.iloc[:, 0].str.contains('weighted avg', case=False, na=False)]
        
        if not weighted_avg.empty:
            result = {
                'pod': pod_name,
                'agent_type': agent_type,
                'model_size': config['model_size'],
                'model_type': config['model_type'],
                'prompting': config['prompting'],
                'precision': weighted_avg['precision'].values[0],
                'recall': weighted_avg['recall'].values[0],
                'f1_score': weighted_avg['f1-score'].values[0],
                'support': weighted_avg['support'].values[0],
                'file': file.name
            }
            all_results.append(result)

# Create DataFrame
df_vuln = pd.DataFrame(all_results)

print(f"\n📊 Loaded {len(df_vuln)} vulnerability detection experiments")
print(f"\nColumns: {list(df_vuln.columns)}")
df_vuln.head(10)

## 3. Data Preprocessing

In [ ]:
# Convert to percentages for better readability
for col in ['precision', 'recall', 'f1_score']:
    df_vuln[f'{col}_pct'] = df_vuln[col] * 100

# Load energy data from emissions.csv files
energy_data = []

for pod_name, config in pod_configs.items():
    pod_dir = RESULTS_DIR / f'runpod_rq2_{pod_name}'
    
    # Handle nested results directories
    emissions_file = pod_dir / 'emissions.csv'
    if not emissions_file.exists():
        emissions_file = pod_dir / 'results' / 'emissions.csv'
    
    if emissions_file.exists():
        df_emissions = pd.read_csv(emissions_file)
        
        # Aggregate energy data
        total_energy = df_emissions['energy_consumed'].sum()  # in kWh
        total_emissions = df_emissions['emissions'].sum()  # in kg CO2
        
        energy_data.append({
            'pod': pod_name,
            'total_energy_kwh': total_energy,
            'total_emissions_kg': total_emissions
        })

df_energy = pd.DataFrame(energy_data)

# Merge energy data with vulnerability results
df_vuln = df_vuln.merge(df_energy, on='pod', how='left')

print("\n✅ Data preprocessing complete!")
print(f"\nDataset shape: {df_vuln.shape}")
df_vuln.head()

## 4. RQ2.1: Dual-Agent vs Multi-Agent Comparison

In [ ]:
# Compare DA vs MA performance
agent_comparison = df_vuln.groupby('agent_type')[['precision_pct', 'recall_pct', 'f1_score_pct']].mean()

print("📊 Dual-Agent vs Multi-Agent Performance:\n")
print(agent_comparison.round(2))

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
agent_comparison.plot(kind='bar', ax=ax, rot=0, color=['#3498DB', '#E74C3C', '#2ECC71'])
ax.set_title('Vulnerability Detection: Dual-Agent vs Multi-Agent', fontsize=14, fontweight='bold')
ax.set_xlabel('Agent Type', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.legend(title='Metrics', labels=['Precision', 'Recall', 'F1 Score'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_da_vs_ma_performance.png', dpi=300, bbox_inches='tight')
plt.show()

# Statistical test
from scipy import stats
da_f1 = df_vuln[df_vuln['agent_type'] == 'Dual-Agent']['f1_score_pct']
ma_f1 = df_vuln[df_vuln['agent_type'] == 'Multi-Agent']['f1_score_pct']
t_stat, p_value = stats.ttest_ind(da_f1, ma_f1)
print(f"\n📊 T-test (DA vs MA F1 Score): t={t_stat:.3f}, p={p_value:.3f}")
if p_value < 0.05:
    print("   ✅ Statistically significant difference!")
else:
    print("   ⚠️  No statistically significant difference")

## 5. RQ2.2: Prompting Strategy Analysis

In [ ]:
# Compare Zero-shot vs Few-shot
prompting_comparison = df_vuln.groupby('prompting')[['precision_pct', 'recall_pct', 'f1_score_pct']].mean()

print("📊 Zero-shot vs Few-shot Performance:\n")
print(prompting_comparison.round(2))

# Visualization
fig, ax = plt.subplots(1, 1, figsize=(10, 6))
prompting_comparison.plot(kind='bar', ax=ax, rot=0, color=['#3498DB', '#E74C3C', '#2ECC71'])
ax.set_title('Vulnerability Detection: Zero-shot vs Few-shot', fontsize=14, fontweight='bold')
ax.set_xlabel('Prompting Strategy', fontsize=12)
ax.set_ylabel('Score (%)', fontsize=12)
ax.legend(title='Metrics', labels=['Precision', 'Recall', 'F1 Score'])
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_prompting_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 6. RQ2.3: Model Configuration Analysis

In [ ]:
# Model size comparison
size_comparison = df_vuln.groupby('model_size')[['precision_pct', 'recall_pct', 'f1_score_pct']].mean()

print("📊 4B vs 30B Model Performance:\n")
print(size_comparison.round(2))

# Model type comparison
type_comparison = df_vuln.groupby('model_type')[['precision_pct', 'recall_pct', 'f1_score_pct']].mean()

print("\n📊 Instruct vs Thinking Performance:\n")
print(type_comparison.round(2))

# Combined visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

size_comparison.plot(kind='bar', ax=axes[0], rot=0, color=['#3498DB', '#E74C3C', '#2ECC71'])
axes[0].set_title('Performance by Model Size', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Model Size', fontsize=12)
axes[0].set_ylabel('Score (%)', fontsize=12)
axes[0].legend(title='Metrics', labels=['Precision', 'Recall', 'F1 Score'])
axes[0].grid(True, alpha=0.3)

type_comparison.plot(kind='bar', ax=axes[1], rot=0, color=['#3498DB', '#E74C3C', '#2ECC71'])
axes[1].set_title('Performance by Model Type', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Model Type', fontsize=12)
axes[1].set_ylabel('Score (%)', fontsize=12)
axes[1].legend(title='Metrics', labels=['Precision', 'Recall', 'F1 Score'])
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_model_configuration_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

## 7. RQ2.4: Energy Efficiency Analysis

In [ ]:
# Energy by agent type
energy_by_agent = df_vuln.groupby('agent_type')[['total_energy_kwh', 'total_emissions_kg']].mean()

print("📊 Energy Consumption by Agent Type:\n")
print(energy_by_agent.round(3))

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

energy_by_agent['total_energy_kwh'].plot(kind='bar', ax=axes[0], rot=0, color=['#3498DB', '#E74C3C'])
axes[0].set_title('Total Energy Consumption', fontsize=14, fontweight='bold')
axes[0].set_xlabel('Agent Type', fontsize=12)
axes[0].set_ylabel('Energy (kWh)', fontsize=12)
axes[0].grid(True, alpha=0.3)

energy_by_agent['total_emissions_kg'].plot(kind='bar', ax=axes[1], rot=0, color=['#3498DB', '#E74C3C'])
axes[1].set_title('Total CO2 Emissions', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Agent Type', fontsize=12)
axes[1].set_ylabel('Emissions (kg CO2)', fontsize=12)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_energy_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

## 8. Comprehensive Heatmap: Configuration × Performance

In [ ]:
# Create pivot table for heatmap
# Create a combined configuration column
df_vuln['config'] = df_vuln['model_size'] + '-' + df_vuln['model_type'] + '-' + df_vuln['prompting']

heatmap_data = df_vuln.pivot_table(
    values='f1_score_pct',
    index='config',
    columns='agent_type',
    aggfunc='mean'
)

print("📊 F1 Score Heatmap (Configuration × Agent Type):\n")
print(heatmap_data.round(2))

# Visualization
plt.figure(figsize=(10, 12))
sns.heatmap(heatmap_data, annot=True, fmt='.2f', cmap='RdYlGn', 
            vmin=40, vmax=70, cbar_kws={'label': 'F1 Score (%)'})
plt.title('RQ2 Vulnerability Detection F1 Score: Configuration × Agent Type', 
          fontsize=14, fontweight='bold')
plt.xlabel('Agent Type', fontsize=12)
plt.ylabel('Configuration', fontsize=12)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'rq2_configuration_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

## 9. Summary & Export

In [ ]:
# Export to Excel
excel_file = OUTPUT_DIR / 'rq2_vulnerability_detection_analysis.xlsx'
with pd.ExcelWriter(excel_file, engine='openpyxl') as writer:
    df_vuln.to_excel(writer, sheet_name='All Results', index=False)
    agent_comparison.to_excel(writer, sheet_name='DA vs MA')
    prompting_comparison.to_excel(writer, sheet_name='Prompting Strategy')
    size_comparison.to_excel(writer, sheet_name='Model Size')
    type_comparison.to_excel(writer, sheet_name='Model Type')
    energy_by_agent.to_excel(writer, sheet_name='Energy Analysis')

print(f"\n✅ Analysis complete!")
print(f"📊 Excel file saved: {excel_file}")
print(f"🖼️  Visualizations saved to: {OUTPUT_DIR}/")

# Display final summary
print("\n" + "="*80)
print("FINAL SUMMARY")
print("="*80)
print(f"\nTotal Experiments Analyzed: {len(df_vuln)}")
print(f"\nBest F1 Score: {df_vuln['f1_score_pct'].max():.2f}% ({df_vuln.loc[df_vuln['f1_score_pct'].idxmax(), 'config']})")
print(f"\nAverage F1 Score by Agent Type:")
print(f"  - Dual-Agent: {df_vuln[df_vuln['agent_type'] == 'Dual-Agent']['f1_score_pct'].mean():.2f}%")
print(f"  - Multi-Agent: {df_vuln[df_vuln['agent_type'] == 'Multi-Agent']['f1_score_pct'].mean():.2f}%")
print(f"\nAverage Energy Consumption: {df_vuln['total_energy_kwh'].mean():.3f} kWh")
print(f"Average CO2 Emissions: {df_vuln['total_emissions_kg'].mean():.3f} kg")
print("\n" + "="*80)

## 10. Key Findings

**To be filled after analysis:**

### RQ2.1: Dual-Agent vs Multi-Agent
- Performance difference: [TBD]
- Precision vs Recall tradeoff: [TBD]

### RQ2.2: Prompting Strategy
- Zero-shot vs Few-shot impact: [TBD]
- Configuration-specific findings: [TBD]

### RQ2.3: Model Configuration
- Model size impact (4B vs 30B): [TBD]
- Reasoning capability impact (Instruct vs Thinking): [TBD]

### RQ2.4: Energy Efficiency
- Energy overhead of multi-agent: [TBD]
- Most efficient configuration: [TBD]